In [2]:
from google.colab import files
uploaded = files.upload()

Saving DATASET.zip to DATASET.zip


In [3]:
import zipfile

with zipfile.ZipFile("DATASET.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [4]:
import os
for c in ["plastic", "cardboard", "metal"]:
    folder = f"data/DATASET/{c}"
    print(c, len(os.listdir(folder)))

plastic 482
cardboard 403
metal 410


In [9]:
import os
import numpy as np
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog
from skimage.filters import sobel

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import seaborn as sns
import matplotlib.pyplot as plt


input_dir = "data/DATASET"
categories = ["plastic", "cardboard", "metal"]

data = []
labels = []

#Feature Extraction


for label in categories:

    path = os.path.join(input_dir, label)

    for file in os.listdir(path):

        if file.lower().endswith((".jpg", ".png", ".jpeg")):

            try:



                img = imread(os.path.join(path, file))


                img = resize(img, (64, 64))


                # HOG Features


                hog_feat = hog(
                    rgb2gray(img),
                    pixels_per_cell=(8,8),
                    cells_per_block=(2,2),
                    orientations=9
                )





                hsv = rgb2hsv(img)

                h_hist = np.histogram(
                    hsv[:,:,0],
                    bins=32,
                    range=(0,1)
                )[0]

                s_hist = np.histogram(
                    hsv[:,:,1],
                    bins=32,
                    range=(0,1)
                )[0]

                v_hist = np.histogram(
                    hsv[:,:,2],
                    bins=32,
                    range=(0,1)
                )[0]

                color_feat = np.hstack([
                    h_hist,
                    s_hist,
                    v_hist
                ])

                edges = sobel(rgb2gray(img))


                edge_small = resize(edges, (16,16))

                edge_feat = edge_small.flatten()

                features = np.hstack([
                    hog_feat,
                    color_feat,
                    edge_feat
                ])

                data.append(features)
                labels.append(label)

            except Exception as e:
                print("Error:", e)



data = np.array(data)
labels = np.array(labels)

print("Total Images:", len(data))
print("Feature Shape:", data.shape)


le = LabelEncoder()
labels = le.fit_transform(labels)

scaler = StandardScaler()

data = scaler.fit_transform(data)

pca = PCA(n_components=300)

data = pca.fit_transform(data)

print("Shape After PCA:", data.shape)


#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)
#Grid Search
params = {

    'C': [0.1, 1, 10, 50],

    'gamma': ['scale', 0.01, 0.001],

    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(

    SVC(
        class_weight='balanced',
        probability=True
    ),

    params,

    cv=3,

    n_jobs=-1
)


grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)
model = grid.best_estimator_
y_pred = model.predict(X_test)
#Evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    average='weighted'
)
recall = recall_score(
    y_test,
    y_pred,
    average='weighted'
)
f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)
print("MODEL EVALUATION")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
#Classification Report
print("Classification Report")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=categories
    )
)

Total Images: 1295
Feature Shape: (1295, 2116)
Shape After PCA: (1295, 300)
Best Parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
MODEL EVALUATION
Accuracy  : 0.8263
Precision : 0.8271
Recall    : 0.8263
F1-Score  : 0.8263
Classification Report
              precision    recall  f1-score   support

     plastic       0.82      0.85      0.84        81
   cardboard       0.86      0.80      0.83        82
       metal       0.81      0.82      0.81        96

    accuracy                           0.83       259
   macro avg       0.83      0.83      0.83       259
weighted avg       0.83      0.83      0.83       259

